# Bloc 3 — Calculateur d'accord entre annotateurs (20 min)

**Situation.** Trois personnes du support étiquettent les **15 mêmes messages** chacune de son côté. Elles ne sont d'accord que sur une partie ; sur *« internet coupe tout le temps, je vais finir par résilier »*, l'une met *technique*, l'autre *résiliation*. Si les humains ne s'accordent pas, par rapport à qui mesure-t-on le modèle ?

C'est votre tour : remplissez la fiche du bloc 3 **seul et en silence**, puis on compare.

Chaque participant étiquette les **15 mêmes messages** (fiche `docs/fiches/fiche_bloc3_annotation`). On saisit les réponses dans `dataset/annotations_atelier.csv` (une colonne par personne : A1…A10, valeurs `facturation`, `technique`, `resiliation`, `livraison`, `autre`).

Tant que ce fichier est vide, le notebook utilise `dataset/annotations_exemple_SIMULEES.csv` : **des annotations simulées**, pas de vraies personnes. Elles servent seulement à montrer les calculs.

**Pourquoi c'est important ?** Si les humains ne sont pas d'accord entre eux, aucun modèle ne peut être « juste » sur ces cas, et l'exactitude mesurée a un plafond.

In [1]:
import itertools
from collections import Counter
from pathlib import Path
import pandas as pd

DATA = Path("..") / "dataset"
CLASSES = ["facturation", "technique", "resiliation", "livraison", "autre"]
reel = pd.read_csv(DATA / "annotations_atelier.csv", dtype=str).fillna("")
cols = [c for c in reel.columns if c.startswith("A") and (reel[c] != "").all()]
if len(cols) >= 2:
    annot, source = reel, "annotations du groupe"
else:
    annot, source = pd.read_csv(DATA / "annotations_exemple_SIMULEES.csv", dtype=str), "annotations SIMULÉES (exemple)"
    cols = [c for c in annot.columns if c.startswith("A")]
messages = pd.read_csv(DATA / "labeling_15.csv")
print(f"Source : {source} — {len(cols)} annotateurs, {len(annot)} messages")
annot[["numero"] + cols]

Source : annotations SIMULÉES (exemple) — 10 annotateurs, 15 messages


,numero,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10
0,1,facturation,facturation,facturation,facturation,facturation,facturation,facturation,facturation,facturation,facturation
1,2,facturation,facturation,facturation,facturation,facturation,facturation,facturation,autre,facturation,facturation
2,3,facturation,facturation,facturation,facturation,facturation,facturation,facturation,facturation,facturation,facturation
3,4,technique,autre,autre,autre,autre,autre,technique,technique,technique,autre
4,5,facturation,facturation,technique,facturation,technique,technique,technique,livraison,technique,facturation
5,6,technique,technique,technique,resiliation,resiliation,technique,resiliation,resiliation,resiliation,technique
6,7,resiliation,facturation,resiliation,resiliation,resiliation,resiliation,facturation,facturation,resiliation,facturation
7,8,resiliation,resiliation,resiliation,resiliation,resiliation,livraison,livraison,technique,resiliation,resiliation
8,9,resiliation,resiliation,resiliation,resiliation,resiliation,resiliation,resiliation,resiliation,resiliation,resiliation
9,10,livraison,facturation,livraison,livraison,autre,livraison,livraison,livraison,livraison,livraison


## 1. Accord brut (en %)
Pour chaque paire de personnes : part des messages où elles ont mis la même étiquette. On fait ensuite la moyenne sur toutes les paires.

In [2]:
import sys; sys.path.insert(0, "..")
from atelier import accord, kappa_cohen, kappa_fleiss   # le code est dans atelier/mesures.py

paires = {(i, j): accord(annot[i], annot[j]) for i, j in itertools.combinations(cols, 2)}
print(f"Accord brut moyen entre deux personnes : {100 * sum(paires.values()) / len(paires):.1f} %")

Accord brut moyen entre deux personnes : 64.9 %


**Question avant la suite** : deux personnes qui cocheraient **au hasard** une des 5 cases, à quelle fréquence tomberaient-elles d'accord ? Et si tout le monde répond « technique » parce que c'est le service le plus fréquent ?
*Réponse attendue* : environ 1 fois sur 5 au hasard, et beaucoup plus si les réponses sont concentrées. Donc 65 % d'accord brut, ce n'est pas 65 % de « vrai » accord.

## 2. Kappa de Cohen (deux personnes)
L'accord brut est flatteur : deux personnes qui répondent au hasard tombent parfois d'accord. Le **kappa** retire cet accord « par chance » :
κ = (accord observé − accord attendu par hasard) / (1 − accord attendu).
Repères usuels (Landis & Koch) : < 0,2 faible · 0,2–0,4 passable · 0,4–0,6 modéré · 0,6–0,8 bon · > 0,8 très bon.

In [3]:
A, B = cols[0], cols[1]
print(f"Cohen κ entre {A} et {B} : {kappa_cohen(annot[A], annot[B]):.3f}  (accord brut {100*accord(annot[A], annot[B]):.0f} %)")
mat = pd.DataFrame([[kappa_cohen(annot[i], annot[j]) for j in cols] for i in cols], index=cols, columns=cols).round(2)
mat

Cohen κ entre A1 et A2 : 0.653  (accord brut 73 %)


,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10
A1,1.00,0.65,0.58,0.58,0.57,0.50,0.50,0.17,0.59,0.66
A2,0.65,1.00,0.57,0.57,0.66,0.50,0.49,0.17,0.43,0.82
A3,0.58,0.57,1.00,0.66,0.67,0.75,0.49,0.16,0.66,0.66
A4,0.58,0.57,0.66,1.00,0.66,0.65,0.57,0.30,0.73,0.74
A5,0.57,0.66,0.67,0.66,1.00,0.58,0.59,0.16,0.66,0.57
A6,0.50,0.50,0.75,0.65,0.58,1.00,0.66,0.32,0.74,0.66
A7,0.50,0.49,0.49,0.57,0.59,0.66,1.00,0.49,0.75,0.66
A8,0.17,0.17,0.16,0.30,0.16,0.32,0.49,1.00,0.47,0.33
A9,0.59,0.43,0.66,0.73,0.66,0.74,0.75,0.47,1.00,0.58
A10,0.66,0.82,0.66,0.74,0.57,0.66,0.66,0.33,0.58,1.00


## 3. Kappa de Fleiss (tout le groupe d'un coup)

In [4]:
kf, accord_par_message = kappa_fleiss(annot[cols])
print(f"Fleiss κ du groupe : {kf:.3f}")

Fleiss κ du groupe : 0.551


## 4. Sur quels messages sommes-nous en désaccord ?

In [5]:
corrige = pd.read_csv(DATA / "labeling_15_corrige.csv").fillna("")
vue = messages.copy()
vue["majorite"] = [Counter(r).most_common(1)[0][0] for r in annot[cols].values]
vue["accord_%"] = [round(100 * Counter(r).most_common(1)[0][1] / len(cols)) for r in annot[cols].values]
vue["reference"] = corrige["label"]
vue["acceptables"] = corrige["labels_acceptables"]
vue["difficulte"] = corrige["difficulte"]
vue.sort_values("accord_%")[["numero", "message", "majorite", "accord_%", "reference", "acceptables", "difficulte"]]

,numero,message,majorite,accord_%,reference,acceptables,difficulte
4,5,l'appli Lumio plante dès que j'ouvre l'onglet ...,technique,50,technique,technique|facturation,piege
5,6,"Mon internet coupe tous les soirs vers 21h, je...",technique,50,technique,technique|resiliation,multi
14,15,lol,autre,50,autre,autre,ambigu
3,4,"Le wifi ne passe pas à l'étage, vous avez des ...",autre,60,technique,technique|autre,ambigu
6,7,"J'ai trouvé moins cher ailleurs, je veux parti...",resiliation,60,resiliation,resiliation|facturation,multi
13,14,Est-ce que je peux mettre mon abonnement en pa...,resiliation,60,autre,autre|resiliation,piege
7,8,"Je veux annuler ma commande passée hier, j'ai ...",resiliation,70,resiliation,resiliation|livraison,ambigu
9,10,Comment je renvoie l'ancien décodeur ? vous m'...,livraison,80,livraison,livraison,facile
1,2,Remboursement des frais de mise en service svp...,facturation,90,facturation,facturation,facile
10,11,"Je ne serai pas là jeudi pour la livraison, on...",livraison,90,livraison,livraison,facile


**À discuter**
- Les désaccords tombent-ils sur les messages « multi » ou « ambigus » ? Ce sont exactement ceux que la politique finale devra **envoyer à un humain**.
- Faut-il une règle d'annotation (« si le client parle d'argent ET de panne, on prend ce qu'il demande en premier ») ? Écrivez-la : elle deviendra une phrase dans vos options.

## 5. Dev vs test scellé
**Question** : si je réécris mes options jusqu'à obtenir 100 % sur ces messages-là, en les regardant à chaque fois, que vaut ce 100 % pour les messages de lundi prochain ?
*Réponse attendue* : pas grand-chose : on a appris *ces* messages. Il faut un paquet qu'on ne regarde pas. Ce paquet a un nom :
- **dev** (`dataset/dev.csv`, 63 messages) : on a le droit de regarder, de se tromper, de réécrire les options. C'est l'atelier de travail.
- **test scellé** (`dataset/test_scelle.csv`, 77 messages) : on ne l'ouvre **qu'une fois**, à la fin, avec des options **figées**. Sinon on finit par ajuster les options au test, et le score ment.
- *Mode rapide* (PC sans GPU Apple ni NVIDIA) : les notebooks utilisent `dev_rapide.csv` (20) et `test_scelle_rapide.csv` (25), tirés de façon stratifiée dans les mêmes fichiers.

### Bonus : wyat comme 11ᵉ annotateur

In [6]:
RUN_WYAT = True   # passez à False si le modèle n'est pas installé
if RUN_WYAT:
    import atelier as at   # (déjà dans le chemin, voir plus haut)
    modele = at.charger_modele()
    decisions = [at.decider(modele, m) for m in messages["message"]]
    vue["wyat"] = [d.selected for d in decisions]
    vue["wyat_marge"] = [round(d.probability_margin, 2) for d in decisions]
    print("wyat = référence :", (vue["wyat"] == vue["reference"]).sum(), "/ 15 ;",
          "wyat = majorité du groupe :", (vue["wyat"] == vue["majorite"]).sum(), "/ 15")
    display(vue[["numero", "message", "majorite", "reference", "wyat", "wyat_marge"]])

Modèle chargé en 2.2 s  (backend=mlx, device=gpu-apple, id=fork-197dbfe273512031109…)
Vitesse mesurée : ≈ 0.16 s par message.


wyat = référence : 9 / 15 ; wyat = majorité du groupe : 9 / 15


,numero,message,majorite,reference,wyat,wyat_marge
0,1,Erreur sur ma facture : vous facturez 2 lignes...,facturation,facturation,facturation,0.98
1,2,Remboursement des frais de mise en service svp...,facturation,facturation,facturation,0.81
2,3,Merci de m'envoyer le détail des communication...,facturation,facturation,autre,0.20
3,4,"Le wifi ne passe pas à l'étage, vous avez des ...",autre,technique,technique,0.88
4,5,l'appli Lumio plante dès que j'ouvre l'onglet ...,technique,technique,facturation,0.99
5,6,"Mon internet coupe tous les soirs vers 21h, je...",technique,technique,resiliation,0.86
6,7,"J'ai trouvé moins cher ailleurs, je veux parti...",resiliation,resiliation,resiliation,0.66
7,8,"Je veux annuler ma commande passée hier, j'ai ...",resiliation,resiliation,livraison,0.95
8,9,"Bonjour, ma box internet ne me sert plus depui...",resiliation,resiliation,technique,0.67
9,10,Comment je renvoie l'ancien décodeur ? vous m'...,livraison,livraison,livraison,0.84


---
**Étape 3, point de contrôle** (`missions/3_accord/MISSION.md`) : le même objectif, à atteindre comme vous voulez (ce notebook, la commande `atelier`, ou un script).
Vérification sur dev : `atelier verifier 3` dans un terminal, ou la cellule ci-dessous.

In [7]:
from pathlib import Path
from atelier.missions import preparer, verifier
preparer(Path("../travail"))   # crée travail/ avec les fichiers de départ s'ils manquent (n'écrase rien)
print(verifier(3, globals().get("modele"), Path("../travail")).texte())   # réutilise le modèle déjà chargé

Étape 3 — Données et accord (bloc 3)
  ✅ 15 messages (15)
  ❌ au moins 2 annotateurs complets (0 : [])
  Pas encore : relisez les indices de MISSION.md, un à la fois (1, puis 2, puis 3).
